In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import re
import json
import os
from dateutil.parser import parse


def clean_date(x):
	if len(re.findall(r"[\d]+", x)) < 1:
		return None

	reg = re.compile(r"[^\w ,/\\\-:+\t\.]+")
	print(x)
	x = re.sub(reg, "", x)
	x = re.sub(r" {2,}", " ", x)
	x = re.sub(r"([pP]ublished|[uU]pdated|[rR]etrieved|[tT]oday)[ ,/\\\-:+\t\.]+", " ", x)
	print(x)
	if len(re.findall(r"(\d){8,10}", x)) > 0:
		x = int(x)
		x = datetime.datetime.utcfromtimestamp(x)
	else:
		x = parse(x)
	return x


def get_year(x):
	if isinstance(x, str):
		if len(re.findall(r"(\d){8,10}", x)) > 0:
			try:
				x = int(x)
				x = datetime.datetime.utcfromtimestamp(x).year
			except:
				return None
		else:
			x = re.findall(r"[0-9]{4,4}", x)
			x = x[0] if len(x) > 0 else None
	else:
		x = [get_year(y) for y in x]
		x = [y for y in x if y is not None]
		x = min(x) if len(x) > 0 else None

	x = int(x) if x is not None and int(x) < 2023 and int(x) > 1989 else None
	return x


def preprocess_reports_dates(x: dict):
	claim_id = list(x.keys())[0]
	reports = list(x.values())[0]
	processesd_dates = []
	for idx, report_date, url in reports:
		new_date = None
		if report_date is not None:
			new_date = get_year(report_date)
		processesd_dates += [(idx, new_date)]

	x[claim_id] = processesd_dates
	return x


all_datasets = [
	"RAWFC_train",
	"RAWFC_test",
	"RAWFC_val",
	"LIAR-RAW_train",
	"LIAR-RAW_test",
	"LIAR-RAW_val",
]
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()
for ax, dataset in zip(axes, all_datasets):
	path = f"dates/{dataset}"
	all_jsons = [
		preprocess_reports_dates(json.load(open(os.path.join(path, c), "r")))
		for c in os.listdir(path)
	]

	num_total_reports = len([
		1 for c in all_jsons for id, reports in c.items() for idx, year in reports
	])
	years = [
		year
		for c in all_jsons
		for id, reports in c.items()
		for idx, year in reports
		if year is not None
	]
	unique_years = set(years)
	num_bins = max(years) - min(years)
	sns.histplot(years, ax=ax, stat="percent", bins=num_bins)
	years_ticks = np.asarray(range(min(years), max(years) + 1))
	years_labels = [year if year in unique_years else "" for year in years_ticks]
	ax.set_xticks(ticks=years_ticks - 0.25, labels=years_labels, rotation=90, size=10)
	ax.set_title(
		f"{dataset}: Number of evidence reports between\n{min(years)}-{max(years)}\nReport coverage {len(years) / num_total_reports * 100:0.2f}%"
	)
plt.tight_layout()
plt.show()

# Entity Exploratory Analysis

In [ ]:
import os
import json
import spacy
from tqdm import tqdm

spacy.prefer_gpu()
parser_spacy = spacy.load("en_core_web_sm")
# parser_spacy = spacy.load("en_core_web_trf", disable=["textcat"])

## LIAR-RAW exploratory analysis

In [ ]:
from collections import defaultdict, Counter


def get_num_relevant(sample: dict):
	is_relevant = 0
	for report in sample["reports"]:
		for tok_sent in report["tokenized"]:
			is_relevant += tok_sent["is_evidence"]
	return is_relevant


def process_reports(sample):
	dependencies = defaultdict(int)
	num_tokens = 0
	report_sentences_batch = [
		tok_sent["sent"] for report in sample["reports"] for tok_sent in report["tokenized"]
	]
	parsed_tok_sent = parser_spacy.pipe(report_sentences_batch)
	for parsed_rep in parsed_tok_sent:
		for tok in parsed_rep:
			dependencies[tok.dep_] += 1
			num_tokens += 1
	dependencies["num_tokens"] = num_tokens
	return dependencies


# @TODO: count all num of tokens to get reference
splits = ["train", "test", "val"][:]
all_deps_list = defaultdict(list)
all_deps = defaultdict(Counter)
relevant_reports = defaultdict(list)
for split in splits:
	data_file = f"../datasets/data/LIAR-RAW/{split}.json"
	output_dir = f"dates/LIAR-RAW_{split}"
	data = json.load(open(data_file, "r"))

	has_relevant = 0
	for idx, sample in tqdm(data):
		num_relevant_sents = get_num_relevant(sample)
		dependencies = process_reports(sample)
		for dep_item, dep_val in dependencies.items():
			all_deps[split][dep_item] += dep_val
		has_relevant += num_relevant_sents > 0
		relevant_reports[split] += [(num_relevant_sents, sample["label"])]
		all_deps_list[split] += [dependencies]

	print(split, f"{has_relevant}/{len(data)} ({has_relevant / len(data):0.3f})")

In [ ]:
import json

split = "train"
data_file = f"../datasets/data/LIAR-RAW/{split}.json"
data = json.load(open(data_file, "r"))

In [ ]:
idx = 30
c = "Health care reform legislation is likely to mandate free sex change surgeries"
for idx in range(0, len(data)):
	if c in data[idx]["claim"]:
		for r in data[idx]["reports"]:
			print(r["link"])
			for s in r["tokenized"][:3]:
				print(s["sent"])
		print("\\textbf{Claim}:", f'"{data[idx]["claim"]}"')
		print("\\textbf{Evidence}:")
		# for s in data[idx]["reports"][3]["tokenized"]:
		# 	print(f'"{s["sent"]}"')
		print("\\textbf{Label}:", data[idx]["label"])
		print("\\textbf{Explanation}:", f'"{data[idx]["explain"]}"')
		# break
		print(f"{idx}-----------------------")

In [ ]:
import os


def load_rawfc_data(split):
	input_path_dir = f"../datasets/data/RAWFC/{split}"
	data = [
		json.load(open(f"{input_path_dir}/{json_file}", "r"))
		for json_file in os.listdir(input_path_dir)
	]
	return data


data = load_rawfc_data("train")
idx = 3
print("\\textbf{Claim}:", f'"{data[idx]["claim"]}"')
print("\\textbf{Explanation}:", f'"{data[idx]["explain"]}"')
print("\\textbf{Label}:", data[idx]["label"])
print("\\textbf{Evidence}:")
for s in data[idx]["reports"][0]["tokenized"]:
	print(f'"{s["sent"]}"')

In [ ]:
no_ev_idx = defaultdict(list)
for split in splits:
	data_file = f"../datasets/data/LIAR-RAW/{split}.json"
	output_dir = f"dates/LIAR-RAW_{split}"
	data = json.load(open(data_file, "r"))

	has_relevant = 0
	for idx, sample in enumerate(tqdm(data)):
		r = 0
		for rep in sample["reports"]:
			r += len([sent["is_evidence"] for sent in rep["tokenized"] if sent["is_evidence"]]) > 0

		has_relevant += r > 0
		if r < 1:
			no_ev_idx[split] += [idx]
	print(split, has_relevant / len(data))

In [ ]:
import json

split = "test"
data_file = f"../datasets/data/LIAR-RAW/{split}.json"
data = json.load(open(data_file, "r"))

In [ ]:
from collections import Counter, defaultdict


def count_claim_labels(data):
	claim_label_counters = Counter()
	for sample in data:
		claim_label_counters[sample["label"]] += 1
	return claim_label_counters

In [ ]:
def load_rawfc_data(split):
	input_path_dir = f"../datasets/data/RAWFC/{split}"
	data = [
		json.load(open(f"{input_path_dir}/{json_file}", "r"))
		for json_file in os.listdir(input_path_dir)
	]
	return data


def load_liaraw_data(split):
	input_path_dir = f"../datasets/data/LIAR-RAW/{split}.json"
	data = json.load(open(input_path_dir, "r"))
	return data


splits = ["train", "test", "val"]
datasets = ["RAWFC", "LIAR-RAW"]
for dataset in datasets:
	per_dataset_info_count_labels = defaultdict(list)
	for split in splits:
		match dataset:
			case "RAWFC":
				load_data_func = load_rawfc_data
			case "LIAR-RAW":
				load_data_func = load_liaraw_data
		data = load_data_func(split)
		print(f"===== {dataset}:{split} =====")
		claim_labels = count_claim_labels(data)
		per_dataset_info_count_labels[dataset] += [claim_labels]
		print(f"Number labels: {claim_labels}; total: {sum(claim_labels.values())}")

	total_count_labels = per_dataset_info_count_labels[dataset][0]
	for count in per_dataset_info_count_labels[dataset][1:]:
		total_count_labels += count
	print(
		f"\n{dataset} total:",
		total_count_labels,
		f"all labels combined: {sum(total_count_labels.values())}",
	)
	print("\n///////////////////////////////////\n")


In [ ]:
def load_rawfc_data(split):
	input_path_dir = f"../datasets/data/RAWFC/{split}"
	data = [
		json.load(open(f"{input_path_dir}/{json_file}", "r"))
		for json_file in os.listdir(input_path_dir)
	]
	return data


def load_liaraw_data(split):
	input_path_dir = f"../datasets/data/LIAR-RAW/{split}.json"
	data = json.load(open(input_path_dir, "r"))
	return data


def count_num_reports(data):
	counter = []
	for sample in data:
		counter += [len(sample["reports"])]
	return counter


splits = ["train", "test", "val"]
datasets = ["RAWFC", "LIAR-RAW"]
for dataset in datasets:
	per_dataset_info = {}
	for split in splits:
		match dataset:
			case "RAWFC":
				load_data_func = load_rawfc_data
			case "LIAR-RAW":
				load_data_func = load_liaraw_data
		data = load_data_func(split)
		print(f"===== {dataset}:{split} =====")
		num_reports = count_num_reports(data)
		per_dataset_info[split] = num_reports
		print(f"Min={min(num_reports)}, Max={max(num_reports)}, Avg={np.mean(num_reports):0.1f}")
	all_splits = []
	for split in per_dataset_info.values():
		all_splits += list(split)

	print("GLOBAL", f"Min={min(all_splits)}, Max={max(all_splits)}, Avg={np.mean(all_splits):0.1f}")

	print("\n///////////////////////////////////\n")

In [ ]:
import os
import json
import numpy as np


def load_rawfc_data(split):
	input_path_dir = f"../datasets/data/RAWFC/{split}"
	data = [
		json.load(open(f"{input_path_dir}/{json_file}", "r"))
		for json_file in os.listdir(input_path_dir)
	]
	return data


def load_liaraw_data(split):
	input_path_dir = f"../datasets/data/LIAR-RAW/{split}.json"
	data = json.load(open(input_path_dir, "r"))
	return data


def count_sent_per_reports(data):
	num_sents = []
	num_reports = []
	for sample in data:
		num_reports += [len(sample["reports"])]
		for report in sample["reports"]:
			num_sents += [len([sent["sent"] for sent in report["tokenized"]])]
	return num_sents, num_reports


splits = ["train", "test", "val"]
datasets = ["RAWFC", "LIAR-RAW"]
per_dataset_info = {}
for dataset in datasets:
	per_split_info = {}
	for split in splits:
		match dataset:
			case "RAWFC":
				load_data_func = load_rawfc_data
			case "LIAR-RAW":
				load_data_func = load_liaraw_data
		data = load_data_func(split)
		print(f"===== {dataset}:{split} =====")
		num_sents, num_reports = count_sent_per_reports(data)
		per_split_info[split] = {"num_sents": num_sents, "num_reports": num_reports}
		print(f"Min={min(num_sents)}, Max={max(num_sents)}, Avg={np.mean(num_sents):0.1f}")
	all_sents = []
	all_reports = []
	for split in per_split_info.values():
		all_sents += split["num_sents"]
		all_reports += split["num_reports"]
	per_dataset_info[dataset] = per_split_info

	print("GLOBAL", f"Min={min(all_sents)}, Max={max(all_sents)}, Avg={np.mean(all_sents):0.1f}")

	print("\n///////////////////////////////////\n")

In [ ]:
import json
import os
import numpy as np


# format: sent_id, entities, entity_types, num_tokens
def traverse_ent_processsed(data):
	num_tokens = []
	num_sents = []
	for sample in data:
		reports = list(sample.values())[0]
		for report in reports:
			sents = list(report.values())[0]
			num_sents += [len(sents)]
			for s in sents:
				*_, tokens = s
				num_tokens += [tokens]
	return num_tokens, num_sents


splits = ["train", "test", "val"]
datasets = ["RAWFC", "LIAR-RAW"]

for dataset in datasets:
	per_split_info = {}
	for split in splits:
		input_dir = os.path.join("spacy_processed", f"{dataset}_{split}")
		data = [
			json.load(open(os.path.join(input_dir, file), "r")) for file in os.listdir(input_dir)
		]
		print(f"===== {dataset}:{split} =====")
		num_tokens, num_sents = traverse_ent_processsed(data)
		per_split_info[split] = {"num_tokens": num_tokens}
		print(f"Min={min(num_tokens)}, Max={max(num_tokens)}, Avg={np.mean(num_tokens):0.1f}")
		per_dataset_info[dataset][split] |= per_split_info[split]
	all_tokens = []
	for split in per_split_info.values():
		all_tokens += split["num_tokens"]

	print("GLOBAL", f"Min={min(all_tokens)}, Max={max(all_tokens)}, Avg={np.mean(all_tokens):0.1f}")

	print("\n///////////////////////////////////\n")

In [ ]:
from collections import defaultdict

val_type_map = {
	"num_sents": "Number of Sentences",
	"num_reports": "Number of Reports",
	"num_tokens": "Number of Tokens",
}

quarts_bounds = [0, 0.25, 0.5, 0.75, 1]
organised_data = defaultdict(lambda: defaultdict(list))
for dataset, split_vals in per_dataset_info.items():
	for split, dict_vals in split_vals.items():
		for val_type, vals in dict_vals.items():
			quarts = np.quantile(vals, quarts_bounds)
			print(quarts)
			val_type = val_type_map[val_type]
			organised_data[val_type]["Split"] += [split.title()] * len(vals)
			organised_data[val_type]["Dataset"] += [dataset] * len(vals)
			organised_data[val_type]["Values"] += vals

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", palette="deep")
fig, axes = plt.subplots(1, len(organised_data.keys()), figsize=(16, 9))
has_legend = True
legend = None
for ax, (val_type, dict_data) in zip(axes, organised_data.items()):
	sns.violinplot(
		data=dict_data,
		x="Split",
		y="Values",
		hue="Dataset",
		ax=ax,
		split=True,
		gap=0.15,
		inner="quart",
		cut=0,
		legend=has_legend,
	)
	if has_legend is True:
		legend = ax.legend()
		has_legend = False
		ax.legend().remove()

	ax.set_ylabel(val_type, size=30)
	ax.set_xlabel("Split", size=30)
	ax.tick_params(axis="both", which="major", labelsize=25)
	for idx, line in enumerate(ax.lines):
		if idx % 3 == 1:
			line.set_color((0, 0, 0))
			line.set_linestyle("-")
		else:
			line.set_color((1, 1, 1))


fig.legend(
	ncols=2,
	handles=legend.legend_handles,
	title="Dataset",
	frameon=False,
	bbox_to_anchor=(0.525, 1.06),
	loc="center",
	fontsize=30,
	title_fontsize=30,
)
plt.tight_layout()
plt.savefig("dataset_statistics.pdf", transparent=True, bbox_inches="tight")
plt.show()

In [ ]:
import json
import os
import numpy as np
from collections import Counter


# format: sent_id, entities, entity_types, num_tokens
def traverse_ent_processsed(data):
	all_ents = Counter()
	all_ent_types = Counter()
	for sample in data:
		reports = list(sample.values())[0]
		for report in reports:
			sents = list(report.values())[0]
			for s in sents:
				sent_id, ents, ent_types, num_tokens = s
				for ent, ent_type in zip(ents, ent_types):
					print(ent, ent_type, s) if ent == "-" else 1
					all_ents[ent] += 1
					all_ent_types[ent_type] += 1

	return all_ents, all_ent_types


splits = ["train", "test", "val"]
datasets = ["RAWFC", "LIAR-RAW"]
per_data_info = {}
for dataset in datasets:
	per_split_info = {}
	for split in splits:
		input_dir = os.path.join("spacy_processed", f"{dataset}_{split}")
		data = [
			json.load(open(os.path.join(input_dir, file), "r")) for file in os.listdir(input_dir)
		]
		print(f"===== {dataset}:{split} =====")
		ents, ent_types = traverse_ent_processsed(data)
		per_split_info[split] = {
			"entities": ents,
			"entity_types": ent_types,
		}
	aux_per_split = {"entities": Counter(), "entity_types": Counter()}
	for split in splits:
		aux_per_split["entities"] += per_split_info[split]["entities"]
		aux_per_split["entity_types"] += per_split_info[split]["entity_types"]
	per_data_info[dataset] = aux_per_split

	print("\n///////////////////////////////////\n")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
datasets = ["RAWFC", "LIAR-RAW"]
fig, axes = plt.subplots(2, 2, figsize=(16, 9))
for ax_row, dataset in zip(axes, datasets):
	topk = 15
	most_common_ents = dict(per_data_info[dataset]["entities"].most_common(topk))
	most_common_ents = {
		ent: ent_count / sum(per_data_info[dataset]["entities"].values()) * 100
		for ent, ent_count in most_common_ents.items()
	}

	most_common_ent_types = dict(per_data_info[dataset]["entity_types"].most_common(topk))
	most_common_ent_types = {
		ent_type: ent_count / sum(per_data_info[dataset]["entity_types"].values()) * 100
		for ent_type, ent_count in most_common_ent_types.items()
	}

	sns.barplot(most_common_ents, ax=ax_row[0])
	sns.barplot(most_common_ent_types, ax=ax_row[1])
	ax_row[0].tick_params(axis="x", rotation=90)
	ax_row[0].set_ylabel("Percentage", fontsize=15)
	ax_row[1].tick_params(axis="x", rotation=90)
	ax_row[0].set_title(dataset, loc="right", fontsize=20, x=1.1, y=1.1)

	ax_row[0].tick_params(axis="both", which="major", labelsize=12)
	ax_row[1].tick_params(axis="both", which="major", labelsize=12)


axes[-1][0].set_xlabel("Entities", fontsize=15)
axes[-1][1].set_xlabel("Type of Entities", fontsize=15)
sns.despine()
plt.tight_layout()
plt.savefig("entities_entitytypes.pdf", transparent=True)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import json
import os
from collections import Counter

sns.set_theme(style="whitegrid")
datasets = ["RAWFC", "LIAR-RAW"]
splits = ["train", "test", "val"]
input_dir = "topics"

per_dataset_info = {}
for dataset in datasets:
	per_split_info = {}
	for split in splits:
		input_path = os.path.join(input_dir, f"{dataset}_{split}", "topics.json")
		topics_data = json.load(open(input_path, "r"))
		topic_sizes = topics_data["topic_sizes"]
		topic_custom_labels = topics_data["custom_labels"]
		per_split_info[split] = {
			", ".join(custom_lab.split(", ")[:2]): count
			for custom_lab, (topic_idx, count) in zip(topic_custom_labels, topic_sizes.items())
		}
	aux_counter = Counter()
	for split in splits:
		aux_counter += Counter(per_split_info[split])
	per_dataset_info[dataset] = aux_counter

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import seaborn as sns

sns.set_theme(style="whitegrid")
datasets = ["RAWFC", "LIAR-RAW"]
fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharey=True)
for ax, dataset in zip(axes, datasets):
	topk = 15

	most_common_topics = dict(per_dataset_info[dataset].most_common(topk))
	most_common_topics = {
		topic: topic_count / sum(per_dataset_info[dataset].values()) * 100
		for topic, topic_count in most_common_topics.items()
	}

	barplot = sns.barplot(most_common_topics, ax=ax)
	all_xticks = barplot.get_xticklabels().copy()
	print(all_xticks)
	print(all_xticks[1])
	for xtick, artist in zip(all_xticks, barplot.get_children()[:-1]):
		if isinstance(artist, Rectangle):
			bar_height = artist.get_height()
			x_coord = artist.get_x()
			bar_text = xtick.get_text()
			bar_text = bar_text.split(", ")
			bar_text = "\n".join(bar_text)
			ax.text(s=bar_text, x=x_coord + 0.1, y=bar_height + 1, rotation=90, fontsize=10)
	ax.set_xticklabels("")
	ax.tick_params(axis="x", rotation=90)
	ax.set_ylabel("Percentage")
	ax.set_title(dataset, fontsize=20)
sns.despine()
plt.tight_layout()
plt.savefig("topics.pdf", transparent=True)
plt.show()

In [ ]:
import pickle

pickle.dump(relevant_reports, open("num_relevant_reports.pkl", "wb"))
pickle.dump(all_deps_list, open("all_deps_list.pkl", "wb"))
pickle.dump(all_deps, open("all_deps.pkl", "wb"))

In [ ]:
import pickle

relevant_reports = pickle.load(open("num_relevant_reports.pkl", "rb"))
all_deps_list = pickle.load(open("all_deps_list.pkl", "rb"))
all_deps = pickle.load(open("all_deps.pkl", "rb"))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame({k: [v] for k, v in all_deps.items()}).T
sns.histplot(data=all_deps)
plt.show()

In [ ]:
from collections import defaultdict, Counter


def get_num_relevant(sample: dict):
	is_relevant = 0
	for report in sample["reports"]:
		for tok_sent in report["tokenized"]:
			is_relevant += tok_sent["is_evidence"]
	return is_relevant


def process_reports(sample):
	dependencies = defaultdict(int)
	atts = defaultdict(list)
	num_tokens = 0
	report_sentences_batch = [
		tok_sent["sent"] for report in sample["reports"] for tok_sent in report["tokenized"]
	]
	parsed_tok_sent = parser_spacy.pipe(report_sentences_batch)
	for parsed_rep in parsed_tok_sent:
		for tok in parsed_rep:
			dependencies[tok.dep_] += 1
			num_tokens += 1
	atts["ents"] = [tok.ent_type_ for rep in parsed_tok_sent for tok in rep]
	dependencies["num_tokens"] = num_tokens
	return dependencies, atts


# @TODO: count all num of tokens to get reference
splits = ["train", "test", "val"][:]
all_deps_list = defaultdict(list)
all_deps = defaultdict(Counter)
other_atts = defaultdict(lambda: defaultdict(list))
relevant_reports = defaultdict(list)
for split in splits:
	data_file = f"../datasets/data/RAWFC/{split}"
	output_dir = f"dates/RAWFC_{split}"
	jsons = [os.path.join(data_file, file) for file in os.listdir(data_file)]

	has_relevant = 0
	has_is_evidence = 0
	for sample_path in tqdm(jsons):
		sample = json.load(open(sample_path, "r"))
		num_relevant_sents = get_num_relevant(sample)
		# dependencies, atts = process_reports(sample)
		# for dep_item, dep_val in dependencies.items():
		# 	all_deps[split][dep_item] += dep_val
		# for att_k, att_v in atts.items():
		# 	other_atts[split][att_k] += [att_v]
		has_relevant += num_relevant_sents > 0
		has_is_evidence += (
			len([
				sent["is_evidence"]
				for rep in sample["reports"]
				for sent in rep["tokenized"]
				if sent["is_evidence"] is True
			])
			> 0
		)
		relevant_reports[split] += [(num_relevant_sents, sample["label"])]
		all_deps_list[split] += [dependencies]

	print(split, f"{has_relevant}/{len(jsons)} ({has_relevant / len(jsons):0.3f})")

In [ ]:
sample

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

drop = ["punct", "prep", "det", "ROOT", "aux", "cc", "mark", "num_tokens"]

sns.set_theme(style="whitegrid")
df = pd.DataFrame({k: [v] for k, v in all_deps["test"].items()}).drop(drop, axis=1).T
df = df.reset_index()
df.columns = ["dep", "count"]
df = df.sort_values(by="count", ascending=False)
df
sns.barplot(df, x="dep", y="count")
plt.xticks(rotation=90)
plt.show()

In [ ]:
s = parser_spacy(sample["reports"][0]["tokenized"][0]["sent"])
list(s.noun_chunks)